# Tool Calling

## Arxiv Tool Calling
langchain_arxiv does not work and has terrible documentation, so using the arxiv library directly instead.

The three main types in the `arxiv` library are as follows:
|Type|Purpose|
|---|---|
|arxiv.Client|Reusable fetcher — holds pagination/retry config and the connection pool. Call client.results(search).|
|arxiv.Search|Describes a query — query, id_list, max_results, sort_by, sort_order.|
|arxiv.Result|A single paper — has .title, .authors, .summary, .published, .pdf_url, .entry_id, etc.|

In [39]:
import arxiv


### Basic Calling

In [ ]:

client = arxiv.Client()
search = arxiv.Search(query="Machine Learning", max_results=2, sort_by=arxiv.SortCriterion.Relevance)

results = client.results(search)

for result in results:
    print(result.title)
    print(result.summary)
    print(result.authors)
    print(result.published)
    print(result.pdf_url)

Changing Data Sources in the Age of Machine Learning for Official Statistics
Data science has become increasingly essential for the production of official statistics, as it enables the automated collection, processing, and analysis of large amounts of data. With such data science practices in place, it enables more timely, more insightful and more flexible reporting. However, the quality and integrity of data-science-driven statistics rely on the accuracy and reliability of the data sources and the machine learning techniques that support them. In particular, changes in data sources are inevitable to occur and pose significant risks that are crucial to address in the context of machine learning for official statistics.
  This paper gives an overview of the main risks, liabilities, and uncertainties associated with changing data sources in the context of machine learning for official statistics. We provide a checklist of the most prevalent origins and causes of changing data sources; no

### Advance query syntax

The `arxiv` library supports advanced query syntax for more precise searches. You can combine author, title, abstract, and other fields using logical operators like `AND`, `OR`, and `NOT`.

Examples:
- `au:del_maestro AND ti:checkerboard` — papers authored by Del Maestro with "checkerboard" in the title.
- `ti:"machine learning" AND abs:"neural networks"` — papers with "machine learning" in the title and "neural networks" in the abstract.
- `au:smith NOT ti:quantum` — papers authored by Smith but not having "quantum" in the title.
- `abs:"deep learning" OR abs:"neural networks"` — papers with "deep learning" or "neural networks" in the abstract.

In [44]:
search = arxiv.Search(query="au:del_maestro AND ti:checkerboard")
first = next(client.results(search))
print(first.title)
print(first.summary)
print(first.authors)
print(first.published)
print(first.pdf_url)


From stripe to checkerboard order on the square lattice in the presence of quenched disorder
  We discuss the effects of quenched disorder on a model of charge density wave (CDW) ordering on the square lattice. Our model may be applicable to the cuprate superconductors, where a random electrostatic potential exists in the CuO2 planes as a result of the presence of charged dopants. We argue that the presence of a random potential can affect the unidirectionality of the CDW order, characterized by an Ising order parameter. Coupling to a unidirectional CDW, the random potential can lead to the formation of domains with 90 degree relative orientation, thus tending to restore the rotational symmetry of the underlying lattice. We find that the correlation length of the Ising order can be significantly larger than the CDW correlation length. For a checkerboard CDW on the other hand, disorder generates spatial anisotropies on short length scales and thus some degree of unidirectionality. We qu

In [ ]:

# from langchain_core.tools import tool
# import arxiv

# @tool
# def arxiv_search(query: str) -> str:
#     """Search arXiv for papers matching the query. Returns titles + summaries."""
#     client = arxiv.Client()
#     search = arxiv.Search(query=query, max_results=2, sort_by=arxiv.SortCriterion.Relevance)
#     docs = []
#     for r in client.results(search):
#         docs.append(f"Title: {r.title}\nPublished: {r.published.date()}\nSummary: {r.summary[:500]}")
#     return "\n\n---\n\n".join(docs) if docs else "No results found."

# # Now this works with .invoke() and agents:
# arxiv_search.invoke("machine learning")

## Wikipedia Tool Calling

### X basic Calling with deprecated langchain libs

In [ ]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

api_wrapper_wiki = WikipediaAPIWrapper(top_k=5, document_content_char_limit=1000)
wiki = WikipediaQueryRun(api_wrapper=api_wrapper_wiki)
result = wiki.invoke("Artificial Intelligence")
print(result)

### Using latest Wikipedia API (wikipedia-api) from PyPI

In [1]:
import wikipediaapi

Key differences between the sync and async API:

- `summary`, `text`, `sections`, `langlinks`, `links`, `backlinks`, `categories`, `categorymembers`, `coordinates`, `images`, `pageid`, `fullurl`, `displaytitle`, … are explicit @property definitions in both APIs. 
In the async API every such property returns a coroutine: await page.summary, await page.sections, await page.pageid, etc.

- `title`, `ns`, `namespace`, `language`, `variant` are plain @property values in both APIs (no await needed).

- `exists()` is a plain method in the sync API; a coroutine method in the async API: await page.exists().

- `section_by_title()` and `sections_by_title()` are plain synchronous methods in both APIs.

In [7]:
# Synchronous client
wiki = wikipediaapi.Wikipedia(user_agent='MyProjectName (merlin@example.com)', language='en')

page_py = wiki.page('Python_(programming_language)')
print("Page - Title: %s" % page_py.title)
# print("Page - Summary: %s" % page_py.summary[0:60])
print(page_py.summary)
print("Page - URL: %s" % page_py.fullurl)

Page - Title: Python_(programming_language)
Python is a high-level, general-purpose programming language that emphasizes code readability, simplicity, and ease-of-writing with the use of significant indentation, an extensive ("batteries-included") standard library, and garbage collection. Python supports multiple programming paradigms but with an emphasis on object-oriented programming and dynamic typing.
Guido van Rossum began working on Python in the late 1980s as a successor to the ABC programming language. Python 3.0, released in 2008, was a major revision and not completely backward-compatible with earlier versions. Beginning with Python 3.5, capabilities and keywords for typing were added to the language, allowing optional static typing. As of 2026, the Python Software Foundation supports Python 3.10, 3.11, 3.12, 3.13, and 3.14, following the project's annual release cycle and five-year support policy. Python 3.15 is currently in the beta development phase, and the stable release

In [8]:
wiki = wikipediaapi.Wikipedia(
    user_agent='MyProjectName (merlin@example.com)',
    language='en',
    extract_format=wikipediaapi.ExtractFormat.WIKI
)

p_wiki = wiki.page("Test 1")
print(p_wiki.text)

In [9]:
wiki_html = wikipediaapi.Wikipedia(
    user_agent='MyProjectName (merlin@example.com)',
    language='en',
    extract_format=wikipediaapi.ExtractFormat.HTML
)
p_html = wiki_html.page("Test 1")
print(p_html.text)

In [5]:
import asyncio

# Asynchronous client
wiki = wikipediaapi.AsyncWikipedia(user_agent='MyProjectName (merlin@example.com)', language='en')
page_py = wiki.page('Python_(programming_language)')
print('Article Summary:', await page_py.summary) # need to use await
print(page_py.title) # cannot use await 
print('Article URL:', await page_py.fullurl) # needs await


Article Summary: Python is a high-level, general-purpose programming language that emphasizes code readability, simplicity, and ease-of-writing with the use of significant indentation, an extensive ("batteries-included") standard library, and garbage collection. Python supports multiple programming paradigms but with an emphasis on object-oriented programming and dynamic typing.
Guido van Rossum began working on Python in the late 1980s as a successor to the ABC programming language. Python 3.0, released in 2008, was a major revision and not completely backward-compatible with earlier versions. Beginning with Python 3.5, capabilities and keywords for typing were added to the language, allowing optional static typing. As of 2026, the Python Software Foundation supports Python 3.10, 3.11, 3.12, 3.13, and 3.14, following the project's annual release cycle and five-year support policy. Python 3.15 is currently in the beta development phase, and the stable release is expected to launch in O

In [11]:
wiki_wiki = wikipediaapi.AsyncWikipedia(
        user_agent='MyProjectName (merlin@example.com)',
        language='en',
        extract_format=wikipediaapi.ExtractFormat.WIKI
    )
page = wiki_wiki.page("Test 1")
text = await page.text
print(text)

# Main Code

In [97]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_ollama import ChatOllama
from langchain_openai import ChatOpenAI
from langchain_openrouter import ChatOpenRouter
from dotenv import load_dotenv

Initialization of LLM

In [98]:
load_dotenv()

# ollama_llm = ChatOllama(model="gemma4:e4b", api_key="OLLAMA_API_KEY")
lmstudio_llm = ChatOpenAI(
    base_url="http://localhost:1234/v1",
    model="microsoft/phi-4-mini-reasoning", 
    api_key="lm_studio",
    )
# openrouter_llm = ChatOpenRouter(model="gpt-oss-20b", api_key="OPENROUTER_API_KEY")


Define the State

In [99]:
# When implementing conversational story, need to use `add_message` method to add messages to the conversation 
# instead of replacing the entire message list.

from langgraph.graph.message import add_messages

class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    
def chat_node(state: ChatState):
    # take user input and add it to the messages list
    messages = state["messages"]
    # send it to the LLM and get the response
    response = lmstudio_llm.invoke(messages)
    # response stored in the messages list
    return {"messages": [response]}

In [100]:
graph = StateGraph(ChatState)

# add  nodes
graph.add_node('chat_node', chat_node)

# add edges
graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', END)

# compile the graph
chatbot = graph.compile()

In [ ]:
# input is given as list of messages, where each message is a dictionary with keys "type" and "content". The "type" can be either "human" or "ai", 
# and the "content" is the text of the message. 
# The initial state of the chatbot is defined as follows:
initial_state = {
    "messages": [HumanMessage(content = "Hello, do you know Rust programming?") ]
}

chatbot.invoke(initial_state)

In [95]:
chatbot.invoke(initial_state)['messages'][-1].content

'\n\nYes, I understand Rust programming. It\'s a systems-oriented language emphasizing **memory safety**, **concurrency**, and **performance** while offering control akin to low-level languages like C. Here\'s an overview:\n\n### Key Features:\n1. **Ownership & Borrowing**:  \n   Rust uses a unique ownership model for memory management, avoiding common pitfalls like null references or data races:  \n   - **Ownership**: A variable owns exclusive rights to data.  \n   - **Borrowing**: References (`&T`) allow read-only access, while mutable borrowing requires exclusive ownership.  \n   - **Lifetimes**: Ensure variables are valid for their intended scope.\n\n2. **Zero-Cost Abstractions**:  \n   Rust combines high-level features (e.g., generics, traits) with low overhead, guaranteeing performance comparable to hand-optimized C/C++ code.\n\n3. **Concurrency**:  \n   Built-in support for parallelism via threads, async/await, and message passing (no shared memory). Example:  \n   ```rust\n   u

In [101]:
while True:
    user_message = input('Type Here: ')
    print('User: ', user_message)
    if user_message.strip().lower() in ['exit', 'quit', 'bye']:
        print('Exiting...')
        break
    response = chatbot.invoke(
        {
            "messages": [HumanMessage(content=user_message)]
        }
    )

    print('AI: ', response['messages'][-1].content)

User:  Hi, I am mike, whats your name?
AI:  

Your name is \boxed{Phi}.
User:  Hi Phi, do you know when your data training cut-off date?
AI:  

The data training cut-off date for Phi, assuming alignment with common industry practices and models similar to predecessors like GPT-3, is **October 2022**. This date marks the last point in time up until which training data was available, ensuring the model remains stable and free from post-release information. However, if Phi underwent additional training or updates after this date (e.g., for performance improvements), the cut-off might extend accordingly. For precise information, official documentation from Microsoft would be required. 

**Final Answer:**  
The data training cut-off date for Phi is \boxed{October 2022}.
User:  do you know the JWT authentication type?
AI:  

Yes, I understand JWT (JSON Web Tokens) authentication. Here's a concise summary:

JWT is an open-standard encoded data structure used for securely transmitting claims b

APIConnectionError: Connection error.